<a href="https://colab.research.google.com/github/diazcas2/mIArmario/blob/main/M%C3%B3dulo_2_Armario.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torch torchvision pillow

In [3]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from PIL import Image
import requests
import torch
import json

model_name = "Qwen/Qwen2-VL-2B-Instruct"
processor = AutoProcessor.from_pretrained(model_name)
model = Qwen2VLForConditionalGeneration.from_pretrained(model_name, torch_dtype=torch.float16).eval()
print("Qwen2 VL cargado correctamente")

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

Qwen2 VL cargado correctamente


In [5]:
!pip install qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 22.4 MB/s eta 0:00:00


In [12]:
from qwen_vl_utils import process_vision_info
import json
import torch
import re

def analizar_prenda(imagen_url, id_prenda):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": imagen_url},
                {"type": "text", "text": """Eres un experto en moda. Analiza la imagen y lista SOLO las prendas principales que se ven claramente, máximo 4.
Los tipos de prenda posibles son: camisa, camiseta, pantalon, vestido, zapatos, corbata, chaqueta, sudadera, jersey, bañador, falda, abrigo, bolso, gorro o lo que consideres oportuno.

Para cada prenda devuelve SOLO una lista JSON con este formato exacto:
[
    {
        "tipo": "camisa",
        "color": "blanco",
        "estampado": "liso",
        "formalidad": "formal",
        "temporada": ["primavera", "verano"]
    }
]
No inventes prendas que no veas claramente. No repitas prendas. No escribas nada más, solo el JSON."""}
            ]
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=1024)

    respuesta = processor.batch_decode(output, skip_special_tokens=True)[0]
    texto = respuesta.split("assistant")[-1].strip().replace("```json", "").replace("```", "").strip()

    if not texto.endswith("]"):
        texto = texto[:texto.rfind("}") + 1] + "]"

    try:
        prendas = json.loads(texto)
    except json.JSONDecodeError:
        objetos = re.findall(r'\{[^{}]+\}', texto)
        prendas = [json.loads(o) for o in objetos]

    resultado = []
    for i, prenda in enumerate(prendas):
        prenda["id"] = f"{id_prenda}_{i+1}"
        prenda["imagen_path"] = imagen_url
        resultado.append(prenda)

    return resultado

# Prueba
url_prueba = "https://images.unsplash.com/photo-1598033129183-c4f50c736f10?w=400"
resultado = analizar_prenda(url_prueba, "prenda_001")
print(json.dumps(resultado, indent=2, ensure_ascii=False))

[
  {
    "tipo": "camisa",
    "color": "blanco",
    "estampado": "liso",
    "formalidad": "formal",
    "temporada": [
      "primavera",
      "verano"
    ],
    "id": "prenda_001_1",
    "imagen_path": "https://images.unsplash.com/photo-1598033129183-c4f50c736f10?w=400"
  }
]


In [ ]:
import json
import os

prendas = [
    {"id": "prenda_001", "url": "https://cdn.grupoelcorteingles.es/statics/manager/contents/images/uploads/2025/03/H1-jwm6c3Jg.jpeg?impolicy=Resize&width=800&height=800"},
    {"id": "prenda_002", "url": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSCj1KAbp_aSRHgRe5p6vRGPO0z2yoJdeQliQ&s"},
    {"id": "prenda_003", "url": "https://images.unsplash.com/photo-1551489186-cf8726f514f8?w=400"},
]

# Cargar armario existente si ya hay uno
if os.path.exists("armario.json"):
    with open("armario.json", "r", encoding="utf-8") as f:
        armario = json.load(f)
    ids_existentes = {p["imagen_path"] for p in armario}
    print(f"Armario existente cargado con {len(armario)} prendas")
else:
    armario = []
    ids_existentes = set()
    print("Armario nuevo creado")

for prenda in prendas:
    if prenda["url"] in ids_existentes:
        print(f"⏭ {prenda['id']} ya está en el armario, saltando...")
        continue
    print(f"Analizando {prenda['id']}...")
    try:
        resultado = analizar_prenda(prenda["url"], prenda["id"])
        for item in resultado:
            armario.append(item)
            print(f"✓ {item['id']} — {item['tipo']} {item['color']}")
    except Exception as e:
        print(f"✗ Error en {prenda['id']}: {e}")

with open("armario.json", "w", encoding="utf-8") as f:
    json.dump(armario, f, indent=2, ensure_ascii=False)

print(f"\nArmario guardado con {len(armario)} prendas")

Armario existente cargado con 3 prendas
Analizando prenda_001...
